# Install

In [ ]:
%%capture
!pip install gradio rapidfuzz
!apt install fonts-noto-cjk

## Tesseract

In [ ]:
%%capture
!apt install tesseract-ocr tesseract-ocr-jpn tesseract-ocr-jpn-vert
!pip install pytesseract

## paddleOCR

In [ ]:
%%capture
import torch

if torch.cuda.is_available():
  !pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/
else:
  !pip install paddlepaddle

!pip install paddleocr

In [ ]:
import paddle
paddle.utils.run_check()

## GLM-OCR

In [ ]:
%%capture
!pip install git+https://github.com/huggingface/transformers.git

## OwOCR

In [ ]:
%%capture
!pip install owocr "owocr[ndlocrlite]" "owocr[meikiocr]" "owocr[mangaocr]" "owocr[easyocr]" "owocr[rapidocr]" protobuf==6.33.1

### Chrome Screen AI

In [ ]:
!sed -i -e "s#f'chromium/third_party/screen-ai/{cipd_platform}'#f'chromium/third_party/screen-ai/{os_name}'#" /usr/local/lib/python3.12/dist-packages/owocr/ocr.py

# Demo

## Functions

In [ ]:
import rapidfuzz

SEP = "   "

def get_score(s1, s2):
  return rapidfuzz.distance.Levenshtein.normalized_similarity(s1, s2)

### Tesseract

In [ ]:
import time
import numpy as np
from pytesseract import pytesseract

def ocr_tesseract(img):
  np_img = np.array(img)
  time_start = time.perf_counter()
  data = pytesseract.image_to_data(np_img, lang="jpn", config='--psm 12', output_type=pytesseract.Output.DATAFRAME)
  time_end = time.perf_counter()

  results = []
  for left, top, width, height, text in  zip(data['left'], data['top'], data['width'], data['height'], data['text']):
    if type(text) == str:
      results.append(text)

  return time_end-time_start, SEP.join(results)

### PaddleOCR

In [ ]:
import time
import numpy as np
from paddleocr import PaddleOCR

ocr_paddle = PaddleOCR(use_angle_cls=True, lang='japan', enable_mkldnn=False)

def ocr_paddleocr(img):
  np_img = np.array(img)
  time_start = time.perf_counter()
  data = ocr_paddle.predict(input=np_img)
  time_end = time.perf_counter()

  results = []
  for (text, poly) in zip(data[0]["rec_texts"], data[0]["rec_boxes"]):
    results.append(text)

  return time_end-time_start, SEP.join(results)

### GLM-OCR

In [ ]:
import time
from transformers import AutoProcessor, AutoModelForImageTextToText

processor_glm = AutoProcessor.from_pretrained("zai-org/GLM-OCR")
model_glm = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path="zai-org/GLM-OCR",
    torch_dtype="auto",
    device_map="auto",
)

def ocr_glmocr(img):
  time_start = time.perf_counter()
  messages = [{"role": "user", "content": [{"type": "image", "url": img}, {"type": "text", "text": "Text Recognition:"}],}]
  inputs = processor_glm.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model_glm.device)
  inputs.pop("token_type_ids", None)
  generated_ids = model_glm.generate(**inputs, max_new_tokens=8192)
  output_text = processor_glm.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
  time_end = time.perf_counter()

  results = []
  for i, text in enumerate(output_text.split('\n')):
    results.append(text)

  return time_end-time_start, SEP.join(results)


### Chrome Screen AI

In [ ]:
import time
from owocr.ocr import ChromeScreenAI

engine_chrome = ChromeScreenAI()
engine_chrome.max_pixel_size = 4096

def ocr_chromeAI(img):
  time_start = time.perf_counter()
  data = engine_chrome(img)
  time_end = time.perf_counter()

  results = []
  for paragraph in data[1].paragraphs:
    for line in paragraph.lines:
      results.append(line.text)

  return time_end-time_start, SEP.join(results)


### meikiocr

In [ ]:
import time
from owocr.ocr import MeikiOCR

engine_meiki = MeikiOCR()

def ocr_meiki(img):
  time_start = time.perf_counter()
  data = engine_meiki(img)
  time_end = time.perf_counter()

  results = []
  for paragraph in data[1].paragraphs:
    for line in paragraph.lines:
      results.append(line.text)

  return time_end-time_start, SEP.join(results)

### NDLOCR-Lite

In [ ]:
import time
from owocr.ocr import NDLOCRLite

engine_ndllite = NDLOCRLite()

def ocr_ndllite(img):
  time_start = time.perf_counter()
  data = engine_ndllite(img)
  time_end = time.perf_counter()

  results = []
  for paragraph in data[1].paragraphs:
    for line in paragraph.lines:
      results.append(line.text)

  return time_end-time_start, SEP.join(results)

### Manga OCR

In [ ]:
import time
from owocr.ocr import MangaOcrSegmented

engine_manga = MangaOcrSegmented()

def ocr_manga(img):
  time_start = time.perf_counter()
  data = engine_manga(img)
  time_end = time.perf_counter()

  results = []
  for paragraph in data[1].paragraphs:
    for line in paragraph.lines:
      results.append(line.text)

  return time_end-time_start, SEP.join(results)

### EasyOCR

In [ ]:
import time
from owocr.ocr import EasyOCR

engine_easy = EasyOCR()

def ocr_easy(img):
  time_start = time.perf_counter()
  data = engine_easy(img)
  time_end = time.perf_counter()

  results = []
  for paragraph in data[1].paragraphs:
    for line in paragraph.lines:
      results.append(line.text)

  return time_end-time_start, SEP.join(results)

### RapidOCR

In [ ]:
import time
from owocr.ocr import RapidOCR

engine_rapid = RapidOCR()

def ocr_rapid(img):
  time_start = time.perf_counter()
  data = engine_rapid(img)
  time_end = time.perf_counter()

  results = []
  for paragraph in data[1].paragraphs:
    for line in paragraph.lines:
      results.append(line.text)

  return time_end-time_start, SEP.join(results)

### OCR

In [ ]:
import unicodedata

ocr_functions = {
    "Tesseract": ocr_tesseract,
    "PaddleOCR": ocr_paddleocr,
    "GLM-OCR": ocr_glmocr,
    "ChromeScreenAI": ocr_chromeAI,
    "meikiocr": ocr_meiki,
    "NDLOCR-Lite": ocr_ndllite,
    "Manga OCR": ocr_manga,
    "EasyOCR": ocr_easy,
    "RapidOCR": ocr_rapid
}

max_size = 1024

def ocr(input_img, txt):
  input_img.thumbnail((max_size, max_size))
  original = unicodedata.normalize('NFKC', SEP.join(txt.split('\n')))
  results = []
  for engine_name in ocr_functions.keys():
    ocr_func = ocr_functions[engine_name]
    time_taken, text_output = ocr_func(input_img)
    score = get_score(original, unicodedata.normalize('NFKC', text_output))
    results.append(f'time: {time_taken}\nscore: {score}\ntext:\n{text_output}')
  return results

## Gradio

In [ ]:
import gradio as gr

css = """
.img_out {height: 600px !important}
.gradio-container textarea {
    field-sizing: content;
}
"""

output_components = []
with gr.Blocks(css=css) as demo:
  with gr.Row():
    input_img = gr.Image(type="pil")
    input_txt = gr.Textbox()
  with gr.Row():
    btn = gr.Button("Exec!")

  for label in ocr_functions.keys():
    with gr.Row():
      # Each engine will have two textboxes: one for time, one for text
      text_output = gr.Textbox(label=label, lines=3, max_lines=100)
      output_components.append(text_output)

  btn.click(fn=ocr, inputs=[input_img, input_txt], outputs=output_components)

demo.launch(debug=False)